# 📈 Market Forward Returns Modeling (Clean)

This notebook provides a clean, leakage-free pipeline to model `forward_returns` using only past information.

**Strategy:**
- Split train.csv into 80% train / 20% validation for model experimentation
- Original test.csv is held out for final evaluation
- Use existing leakage-free features (already lagged)
- Handle missing values via median imputation
- Scaling, feature selection, Random Forest training
- Evaluation on validation set

## 1) Setup


In [6]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression

root = Path('.')



## 2) Load Data


In [7]:
# Load training data only (test.csv held out for final evaluation)
train_full = pd.read_csv(root / 'train.csv')

print('Full training data shape:', train_full.shape)
print('Date range:', train_full['date_id'].min(), '-', train_full['date_id'].max())

# Split into 80% train / 20% validation based on time
split_idx = int(len(train_full) * 0.8)
train_df = train_full.iloc[:split_idx].copy()
val_df = train_full.iloc[split_idx:].copy()

print(f'\n80/20 Time-based Split:')
print(f'Train: {len(train_df)} samples (date_id {train_df["date_id"].min()} - {train_df["date_id"].max()})')
print(f'Validation: {len(val_df)} samples (date_id {val_df["date_id"].min()} - {val_df["date_id"].max()})')
print(f'\nOriginal test.csv will be used for final model evaluation')

Full training data shape: (8990, 98)
Date range: 0 - 8989

80/20 Time-based Split:
Train: 7192 samples (date_id 0 - 7191)
Validation: 1798 samples (date_id 7192 - 8989)

Original test.csv will be used for final model evaluation


## 3) Feature Selection and Data Preparation

In [8]:
# Use the existing features from the dataset
# These features are already lagged and leakage-free

# Identify feature columns (exclude target, date_id, and metadata columns)
exclude_cols = ['forward_returns', 'date_id', 'is_scored', 'risk_free_rate', 'market_forward_excess_returns']
feature_cols = [col for col in train_df.columns if col not in exclude_cols]

print(f"Total features available: {len(feature_cols)}")
print(f"Sample features: {feature_cols[:10]}")

# Check for missing values
null_counts_train = train_df[feature_cols].isnull().sum()
print(f"\nTrain features with missing values: {(null_counts_train > 0).sum()}")
print(f"Train features without missing values: {(null_counts_train == 0).sum()}")

# Strategy: Fill missing values with median from training set
# This is a simple imputation strategy that doesn't leak information
train_df2 = train_df.copy()
val_df2 = val_df.copy()

for col in feature_cols:
    if col in train_df2.columns and train_df2[col].isnull().any():
        median_val = train_df2[col].median()
        train_df2[col] = train_df2[col].fillna(median_val)
        if col in val_df2.columns:
            val_df2[col] = val_df2[col].fillna(median_val)

# Remove any remaining rows with NaN in target
train_df2 = train_df2.dropna(subset=['forward_returns'])
val_df2 = val_df2.dropna(subset=['forward_returns'])

print(f"\nFinal data shapes:")
print(f"Train: {len(train_df2)} rows")
print(f"Validation: {len(val_df2)} rows")

assert len(train_df2) > 0, "No training data available"
assert len(val_df2) > 0, "No validation data available"

Total features available: 94
Sample features: ['D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'E1']

Train features with missing values: 85
Train features without missing values: 9

Final data shapes:
Train: 7192 rows
Validation: 1798 rows


## 4) Scale, Select Features, Train RF

In [9]:
# Prepare train and validation sets
X_train = train_df2[feature_cols].values
y_train = train_df2['forward_returns'].values
X_val = val_df2[feature_cols].values
y_val = val_df2['forward_returns'].values

print(f'Train/Validation shapes: X_train={X_train.shape}, X_val={X_val.shape}')

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Feature selection: select top k features based on f_regression
k = min(20, X_train.shape[1])
selector = SelectKBest(score_func=f_regression, k=k)
X_train_sel = selector.fit_transform(X_train_scaled, y_train)
X_val_sel = selector.transform(X_val_scaled)

# Get selected feature names
selected_mask = selector.get_support()
selected_features = [f for f, selected in zip(feature_cols, selected_mask) if selected]
print(f"\nSelected {k} features:")
for i, feat in enumerate(selected_features[:10]):
    print(f"  {i+1}. {feat}")
if len(selected_features) > 10:
    print(f"  ... and {len(selected_features) - 10} more")

# Train Random Forest
print("\nTraining Random Forest...")
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train_sel, y_train)

y_train_pred = rf.predict(X_train_sel)
y_val_pred = rf.predict(X_val_sel)
print("Training complete!")

Train/Validation shapes: X_train=(7192, 94), X_val=(1798, 94)

Selected 20 features:
  1. D1
  2. D2
  3. D4
  4. D8
  5. E11
  6. E12
  7. E19
  8. I2
  9. M1
  10. M17
  ... and 10 more

Training Random Forest...
Training complete!


## 5) Evaluation


In [10]:
def metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        'r2': r2_score(y_true, y_pred),
        'rmse': float(np.sqrt(mse)),
        'mae': mean_absolute_error(y_true, y_pred),
    }

train_metrics = metrics(y_train, y_train_pred)
val_metrics = metrics(y_val, y_val_pred)

print("\n🌲 Leakage-free Random Forest Model")
print("=" * 50)
print(f"Features used: {k} / {X_train.shape[1]}")
print(f"\nTRAIN SET ({len(y_train)} samples):")
print(f"  R²:   {train_metrics['r2']:.6f}")
print(f"  RMSE: {train_metrics['rmse']:.6f}")
print(f"  MAE:  {train_metrics['mae']:.6f}")

print(f"\nVALIDATION SET ({len(y_val)} samples):")
print(f"  R²:   {val_metrics['r2']:.6f}")
print(f"  RMSE: {val_metrics['rmse']:.6f}")
print(f"  MAE:  {val_metrics['mae']:.6f}")

# Analyze predictions vs actuals
print(f"\nValidation Predictions Analysis:")
print(f"  Actual returns - Mean: {y_val.mean():.6f}, Std: {y_val.std():.6f}")
print(f"  Predicted returns - Mean: {y_val_pred.mean():.6f}, Std: {y_val_pred.std():.6f}")
print(f"  Prediction range: [{y_val_pred.min():.6f}, {y_val_pred.max():.6f}]")
print(f"  Actual range: [{y_val.min():.6f}, {y_val.max():.6f}]")

# Direction accuracy (sign prediction)
correct_direction = np.sign(y_val) == np.sign(y_val_pred)
print(f"\nDirectional Accuracy: {correct_direction.mean():.2%}")


🌲 Leakage-free Random Forest Model
Features used: 20 / 94

TRAIN SET (7192 samples):
  R²:   0.211860
  RMSE: 0.009242
  MAE:  0.006777

VALIDATION SET (1798 samples):
  R²:   -0.035734
  RMSE: 0.011286
  MAE:  0.007954

Validation Predictions Analysis:
  Actual returns - Mean: 0.000612, Std: 0.011089
  Predicted returns - Mean: 0.000936, Std: 0.002166
  Prediction range: [-0.008249, 0.019071]
  Actual range: [-0.039754, 0.040661]

Directional Accuracy: 52.00%
